# 수험번호를 반드시 입력하세요!

**문제를 시작하기 전에, 바로 아래 셀의 `student_id`에 본인의 수험번호와 본인의 LLM API를 입력하고 실행하세요.**

예: `student_id = '001234'` — 앞자리 0도 포함하여 따옴표 안에 입력하세요.


In [1]:
# 본인의 수험번호를 따옴표 안에 입력하세요. 예: '001234'
student_id = '000000'   # [정답안] 예시 번호
if not student_id or not student_id.isascii() or not student_id.isdigit():  # 비어 있거나, 영숫자 외 문자가 있거나, 숫자가 아니면 오류
    raise ValueError('수험번호를 숫자로만 입력하세요. 앞자리 0도 그대로 적습니다.')  # 수험번호 형식 오류를 알리고 실행을 멈춤
print('수험번호:', student_id)  # 입력한 수험번호를 화면에 출력

수험번호: 000000


In [2]:
# 본인의 API 키를 따옴표 안에 입력하고 실행하세요. 이후 모델 호출에 사용됩니다.
USER_API_KEY = 'dummy'  # API 키 ('dummy'면 모의 모드)
if not isinstance(USER_API_KEY, str) or not USER_API_KEY.strip():  # 문자열이 아니거나 비어 있으면
    raise ValueError('USER_API_KEY에 본인의 API 키를 입력하세요.')  # 오류
USER_API_KEY = USER_API_KEY.strip()  # 앞뒤 공백 제거


## 코드 작성 안내
- 답안 작성과 관련 있는 셀은 앞쪽에 `답안 작성 셀`로 시작하는 마크다운 셀이 있습니다.
- 수정이 불가한 셀에는 `# 수정 금지`로 표시가 되어있습니다. 해당 주석이 있는 셀의 코드는 절대 수정하지 마세요.
- 수정 가능한 셀은 `# 수정 가능`으로 표시가 되어있습니다. 다만 셀 안에서 함수명 등 변경하면 안되는 부분도 있으니, 그런 경우 해당 셀 혹은 문항의 안내를 참고하세요.

# 문제 3. 언어모델 Prompting 기반 지능형 함수 구현 (예측)

> **[예측 문제]** 공식 샘플(전세계약 녹취 → 계약서)과 같은 형식으로 영역만 바꾼 연습 문제입니다. 실제 시험 문제가 아닙니다.

### 배경
입력 `x`를 지능적으로 처리해 출력 `y`를 내는 함수 `f()`를 **한글 시스템 프롬프트만으로** 구현합니다. (`y = f(x)`)

### 문제 지시문
온라인 패션 쇼핑몰 고객센터의 **반품·교환 상담 녹취(전화/채팅)** 에서 접수서에 필요한 정보만 뽑아, 정해진 JSON 형식으로 출력하는 함수를 "한글"로 구현하세요.\
녹취에는 고객이 말을 바꾸는 경우, 상담원이 정정하는 경우, 받아들여지지 않은 요청이 섞여 있습니다.

### 접수 규정 (서비스 팀 제공)
- **요청유형**: `반품`(환불) 또는 `교환`. 상담 중 바뀌면 최종 요청을 씁니다.
- **사유유형**: `단순변심`, `사이즈불일치`, `상품불량`, `오배송` 중 하나. 판매자 확인 결과가 고객 주장과 다르면 판매자 확인 결과를 따릅니다.
- **반품배송비_원**: 고객 사유(`단순변심`, `사이즈불일치`)는 왕복 **6,000원**, 판매자 사유(`상품불량`, `오배송`)는 **0원**.
- **결제금액_원**: 반품·교환 **대상 수량만큼의** 실제 결제금액(쿠폰·할인 반영 후).
- **환불예정금액_원**: 반품이면 `결제금액_원 - 반품배송비_원`, 교환이면 `null`.
- **교환사이즈/교환색상**: 교환일 때만 값이 있고, 바꾸지 않는 항목은 주문 값 그대로. 반품이면 둘 다 `null`.
- **상품분류**: 아래 20개 영문 카테고리 중 하나.
  `Tshirts, Shirts, Casual Shoes, Watches, Sports Shoes, Kurtas, Tops, Handbags, Heels, Sunglasses, Wallets, Flip Flops, Sandals, Briefs, Belts, Backpacks, Socks, Formal Shoes, Perfume and Body Mist, Jeans`
- **회수 주소**: 실제로 상품을 회수할 주소(고객이 바꾸면 바꾼 주소).

- 수험번호 입력 후, 답안은 `system_prompts()` 함수 안에서만 작성하세요. 모델 호출·검증·저장 코드는 수정하지 마세요.
- **제공된 연결 파일과 호출 코드는 평가 조건이므로 변경할 수 없습니다.** 다른 목적으로 LLM을 쓰면 부정행위 처리될 수 있습니다.
- 모든 샘플에 적용할 단일 프롬프트를 작성하세요. 문항별 정답을 넣거나 함수 안에서 모델을 별도로 호출하지 마세요.

> ## [예측 문제 정답안]
> 반품·교환 접수서 22개 항목을 뽑는 단일 시스템 프롬프트

## 환경 확인

먼저 아래 셀로 노트북 커널을 확인하세요. 이 폴더의 `.venv\Scripts\python.exe`를 선택해야 합니다.\
필요한 패키지는 `setup.cmd`가 설치합니다. 


In [3]:
# 수정 금지
# 먼저 현재 커널과 OpenAI SDK를 확인하세요.

import sys  # 파이썬 인터프리터 정보(sys.prefix, sys.executable)를 쓰기 위한 모듈
from pathlib import Path  # 경로를 객체로 다루는 Path 클래스

print('Python:', sys.version.split()[0])  # 현재 파이썬 버전 출력 (예: 3.12.10)
print('현재 커널:', sys.executable)  # 지금 노트북을 실행 중인 python.exe 경로 출력 → .venv인지 확인
print('작업 폴더:', Path.cwd())  # 현재 작업 폴더 출력 (상대 경로 data/... 의 기준)
expected_env = Path.cwd() / '.venv'  # 작업 폴더 아래 .venv 폴더 경로를 기대값으로 만듦
if Path(sys.prefix).resolve() != expected_env.resolve():  # 실행 중인 파이썬의 환경 폴더가 기대한 .venv와 다르면
    raise RuntimeError(f"{expected_env / 'Scripts' / 'python.exe'}를 노트북 커널로 선택하세요.")  # 잘못된 커널 오류
try:  # 시도
    import openai  # OpenAI 파이썬 SDK
except ImportError as error:  # SDK가 없으면
    raise RuntimeError('OpenAI SDK가 없습니다. setup.cmd로 환경을 준비하고 커널을 다시 시작하세요.') from error  # 안내 오류
print('OpenAI SDK:', openai.__version__)  # SDK 버전 출력
print('커널 확인 완료. 다음 셀로 진행하세요.')  # 확인 메시지


Python: 3.12.10
현재 커널: C:\Users\SSAFY\Desktop\AI_Lv3_sample_solution\.venv\Scripts\python.exe
작업 폴더: C:\Users\SSAFY\Desktop\AI_Lv3_final_predict


OpenAI SDK: 3.46.0
커널 확인 완료. 다음 셀로 진행하세요.


In [4]:
# 수정 금지

import csv  # CSV 읽기·쓰기
from datetime import date  # 날짜 형식 검사용
import json  # JSON 파일(classes.json) 읽기용
import math  # cos 계산용
from pathlib import Path  # 경로를 객체로 다루는 Path 클래스
import pickle  # gms_config.pkl(연결 객체) 읽기용
import time  # 학습 시간 측정용


## 데이터 예시 및 응답 Schema

노트북과 같은 폴더의 `F3_qa.csv`에 프롬프트 테스트용 예제 2개가 제공됩니다. 한 행이 한 문제이며 `task`, `question` 두 열만 있습니다. 정답은 공개하지 않습니다.

In [5]:
# 수정 금지

def load_questions(path):  # CSV를 문항 목록으로 읽는 함수
    """task, question 두 열이 있는 CSV를 문항 목록으로 읽습니다."""  # 함수 설명(docstring)
    with Path(path).open(encoding='utf-8-sig', newline='') as source:  # utf-8-sig: 엑셀이 붙이는 BOM 문자도 처리
        reader = csv.DictReader(source)  # CSV를 행 사전으로 읽음
        if reader.fieldnames != ['task', 'question']:  # 열 이름이 정확히 task, question인지
            raise ValueError('CSV는 task, question 두 열이어야 합니다.')  # 아니면 오류
        questions = [  # 문항 목록 만들기
            {'task': row['task'], 'question': row['question'].replace('\\n', '\n')}  # CSV 안의 글자 '\n'을 실제 줄바꿈으로 되돌림
            for row in reader  # 모든 행에 대해
        ]
    if not questions:  # 문항이 없으면
        raise ValueError('CSV에 문항이 없습니다.')  # 오류
    return questions  # 문항 목록 반환

examples = load_questions('F3_qa.csv')  # F3 예제 2개 읽기
print('예제 수:', len(examples))  # 개수 출력

예제 수: 2


### 예시 1: 반품·교환 접수 정보 추출

상담 일시: 2030-05-12 / 쇼핑몰 고객센터 전화 녹취

[상담원] 네, 고객님. 주문번호 먼저 불러 주시겠어요?

[고객] 에이, 이공삼공공오공팔, 하이픈, 공공일칠이요. 아 잠깐만요, 이건 지난번 주문이고요. 에이 이공삼공공오공구 하이픈 공공사이입니다.

[상담원] A20300509-0042 확인됐습니다. 김도현 고객님 맞으시죠?

[고객] 네 맞아요. 연락처는 공일공 사칠팔구 일이삼사로 남겨 주세요. 주문할 때 적은 번호는 회사 대표번호라서요.

[상담원] 010-4789-1234로 기록하겠습니다. 어떤 상품 문의세요?

[고객] 러닝화요. 에어웨이브 러너2. 270 신었는데 발볼이 좁아서 앞이 너무 꽉 껴요.

[상담원] 운동화 상품이고, 검정 270 한 켤레 맞으신가요? 결제는 팔만 구천 원으로 되어 있습니다.

[고객] 네. 환불은 말고 바꾸고 싶어요. 280으로요. 아니다, 매장 가서 신어 봤는데 275가 딱이었어요. 275로 해 주세요.

[상담원] 색상도 바꾸시겠어요?

[고객] 회색도 예쁘던데 고민되네요. 그냥 같은 검정으로 할게요.

[상담원] 사이즈 교환은 고객님 사유라 왕복 배송비 육천 원이 발생합니다.

[고객] 신발 자체가 원래 작게 나온 거 아니에요? 그럼 불량 아닌가요?

[상담원] 확인해 보니 정사이즈 상품이고 같은 문의나 불량 접수가 없어서요. 죄송하지만 사이즈 사유로 접수됩니다.

[고객] 알겠어요. 그럼 그렇게 할게요.

[상담원] 회수는 주문하신 배송지인 서울 마포구 월드컵북로 사백으로 가면 될까요?

[고객] 아뇨, 요즘 회사에서 받아요. 경기도 성남시 분당구 판교역로 이백삼십오, 비동 칠층이요.

[상담원] 판교역로 235, B동 7층 맞으시죠.

[고객] 네. 수거는 내일은 출장이라 안 되고 모레 오전에 와 주세요.

[상담원] 네, 말씀하신 날로 회수 접수했습니다.

### 예시 2: 반품·교환 접수 정보 추출

접수일: 2030-06-03 / 채팅 상담 기록

고객: 안녕하세요 주문한 거 반품하려고요. 주문번호 B20300528-1187

상담원: 확인했습니다. 이수진 고객님, 주문 상품이 린넨 오버핏 셔츠 화이트 M 두 장, 데님 와이드 팬츠 한 장이네요. 어떤 상품 문의실까요?

고객: 셔츠요. 두 장 시켰는데 한 장이 화이트가 아니라 아이보리가 왔어요.

상담원: 보내 주신 사진 확인했습니다. 화이트 주문인데 아이보리가 발송되었네요, 저희 출고 실수입니다. 같은 화이트로 교환해 드릴까요?

고객: 다시 받는 건 필요 없고, 그 한 장만 환불해 주세요.

상담원: 셔츠 한 장 결제금액은 사만 이천 원입니다. 두 장 합쳐서 팔만 사천 원 결제하셨어요.

고객: 네 한 장만요. 바지는 그냥 입을게요.

상담원: 출고 실수라 배송비는 없습니다. 연락처는 주문서 번호로 할까요?

고객: 네 그대로요. 공일공 이이삼삼 오오육육

상담원: 회수지는 주문 배송지 부산광역시 해운대구 센텀중앙로 칠십구, 센텀아파트 백이동 천삼백일호 맞으세요?

고객: 네 거기요. 아 천삼백일호 아니고 천삼백십일호예요. 주문할 때 잘못 썼네요.

상담원: 1311호로 수정했습니다. 회수는 언제가 편하세요?

고객: 이번 주 금요일이요.

상담원: 금요일 회수로 접수됐습니다.

### **지켜야 할 응답 형식 (중요!)**

시스템 프롬프트는 언어모델이 녹취 하나를 입력받아, 아래 모든 항목을 포함한 **JSON 객체 하나**를 반환하도록 유도해야 합니다.

| 항목 | Python 값의 형식 | 예시 |
| --- | --- | --- |
| 이름·상품명·주소·사이즈·색상 | `str` | `'홍길동'`, 사이즈 `'270'`, `'M'`, 색상 `'검정'`, 상세주소 `'101동 1001호'` |
| 주문번호 | `str` (영문 대문자, 하이픈 포함) | `'A20300101-0001'` |
| 연락처 | `str` `'010-0000-0000'` 형식 | `'010-1234-5678'` |
| 금액 | 원 단위 `int` | `39000` |
| 수량 | `int` | `1` |
| 날짜 | `str` `YYYY-MM-DD` | `'2030-01-02'` |
| 정해진 값 중 하나 | `str` (Schema의 `enum`) | `'교환'` |
| 확인할 수 없거나 해당 없는 값 | `None` (JSON `null`) | `None` |

- 문자열 `"None"`, `"null"`을 넣지 마세요. 날짜는 0을 생략하지 않고, 존재하는 날짜여야 합니다.
- `enum`이 있는 항목은 목록의 값과 **글자까지 똑같아야** 합니다(공백·대소문자 포함).

In [6]:
# 수정 금지

CLASS_NAMES = ["Tshirts", "Shirts", "Casual Shoes", "Watches", "Sports Shoes", "Kurtas", "Tops", "Handbags", "Heels", "Sunglasses", "Wallets", "Flip Flops", "Sandals", "Briefs", "Belts", "Backpacks", "Socks", "Formal Shoes", "Perfume and Body Mist", "Jeans"]  # 상품분류로 허용되는 20개 이름

def _s():  # '문자열 또는 null' 규칙을 만드는 짧은 함수
    return {'type': ['string', 'null']}  # 규칙 반환

response_schema = {  # 응답 JSON의 키와 자료형 정의 (JSON Schema)
    'type': 'object',  # 최상위는 객체
    'properties': {  # 키별 규칙
        '접수일': {'type': ['string', 'null'], 'format': 'date'},  # 날짜
        '주문번호': _s(),  # 문자열 또는 null
        '주문자': _s(),  # 문자열 또는 null
        '연락처': _s(),  # 문자열 또는 null
        '상품명': _s(),  # 문자열 또는 null
        '상품분류': {'type': ['string', 'null'], 'enum': CLASS_NAMES + [None]},  # 20개 이름 중 하나 또는 null
        '주문사이즈': _s(),  # 문자열 또는 null
        '주문색상': _s(),  # 문자열 또는 null
        '수량': {'type': ['integer', 'null'], 'minimum': 1},  # 정수, 1 이상
        '결제금액_원': {'type': ['integer', 'null'], 'minimum': 0},  # 정수, 0 이상
        '요청유형': {'type': ['string', 'null'], 'enum': ['반품', '교환', None]},  # '반품' / '교환' 중 하나
        '사유유형': {'type': ['string', 'null'], 'enum': ['단순변심', '사이즈불일치', '상품불량', '오배송', None]},  # 네 가지 사유 중 하나
        '교환사이즈': _s(),  # 문자열 또는 null
        '교환색상': _s(),  # 문자열 또는 null
        '반품배송비_원': {'type': ['integer', 'null'], 'minimum': 0},  # 정수, 0 이상
        '환불예정금액_원': {'type': ['integer', 'null'], 'minimum': 0},  # 정수 또는 null
        '회수시도': _s(),  # 문자열 또는 null
        '회수시군구': _s(),  # 문자열 또는 null
        '회수도로명': _s(),  # 문자열 또는 null
        '회수건물번호': _s(),  # 문자열 또는 null
        '회수상세주소': _s(),  # 문자열 또는 null
        '회수희망일': {'type': ['string', 'null'], 'format': 'date'},  # 날짜
    },
    'additionalProperties': False,  # 정의되지 않은 키는 허용 안 함
}
response_schema['required'] = list(response_schema['properties'])  # 모든 키를 필수로 지정

# **답안 작성 셀 - 시스템 프롬프트 작성**

`system_prompts()` 함수를 완성하세요. 입력 인자는 없으며, 반환값은 **모든 문항에 공통으로 적용할 시스템 프롬프트 문자열 단 하나**입니다.\
역할, 추출 규칙, 출력 형식을 이 함수 안에서 작성하세요.

### API에 전달되는 메시지

| API 역할 | 전달되는 내용 | 작성 방법 |
| --- | --- | --- |
| `system` | `system_prompts()`의 반환값 | 함수 안에서 공통 지시문 작성 (응시자가 작성해야하는 부분) |
| `user` | `problem['question']` | 제공 코드가 해당 문항의 녹취 원문을 그대로 전달 |

`question`을 별도 프롬프트로 만들거나 시스템 프롬프트에 붙이는 것은 금지됩니다.\
아래 제공된 호출 코드는 시스템 지시문과 녹취를 두 메시지로 구분해 전달합니다.\
각 요청은 독립적이므로 이전 문항의 대화에 의존하지 마세요.

### JSON Schema에 맞춰 응답받기

위 `response_schema`는 참고용으로 정의되었습니다. 제공된 코드에서는 언어모델에 전달되지 않습니다.\
이 스키마를 사용하고 싶다면, `json.dumps(response_schema, ensure_ascii=False)`로 문자열을 만들어 시스템 프롬프트에 넣을 수 있습니다.\
스키마와 함께 **어떤 응답을 원하는지**도 구체화해서 프롬프트를 만드세요. 출력 지시의 예시는 다음과 같습니다.

> 아래 JSON Schema에 맞는 JSON 객체 하나만 출력하세요. 모든 키를 포함하고, 설명이나 코드 블록은 붙이지 마세요.

모델이 반환할 것은 Schema 자체가 아니라 **추출한 값이 채워진 객체**입니다.\
키와 문자열에는 큰따옴표를 쓰고, 미확정 값은 JSON의 `null`로 표현하도록 안내하세요. 숫자·날짜 형식은 앞의 표를 따릅니다.\
이 문제는 API의 Structured Outputs나 JSON mode를 설정하지 않습니다. 프롬프트로 형식을 요청한 뒤, 제공 코드로 검증합니다.

### 공식 문서

- [OpenAI Python SDK 사용법](docs/pages/developers.openai.com/api_reference_python--7362aaaa36b8.html): `OpenAI` 객체와 `chat.completions.create` 호출 예시
- [Chat Completions API](docs/pages/developers.openai.com/api_reference_resources_chat--ef63818ccfcb.html): `messages`, 역할과 응답 구조
- [OpenAI Prompt engineering](docs/pages/developers.openai.com/api_docs_guides_prompt-engineering--fcee37c55ff3.html): 지시문, 문맥과 출력 형식 작성
- [Structured Outputs](docs/pages/developers.openai.com/api_docs_guides_structured-outputs--7c7a6f9a9831.html): API에서 출력 형식을 지정하는 별도 기능. 이 문제에서는 사용하지 않습니다.

문서의 호출 예시는 참고용입니다. 제공된 연결 파일과 호출 코드를 수정하거나 별도 클라이언트로 우회 호출하면 안 됩니다.

# **답안 작성 셀**

In [7]:
# 수정 가능
# 함수명은 변경하지 마세요.

def system_prompts():  # 시스템 프롬프트를 만드는 함수 (이름 고정, 인자 없음)
    """[정답안] 반품·교환 접수서 추출용 단일 시스템 프롬프트"""  # 함수 설명(docstring)
    schema_text = json.dumps(response_schema, ensure_ascii=False)  # 스키마를 한글 그대로(ensure_ascii=False) JSON 문자열로
    keys_text = ', '.join(response_schema['properties'])  # 키 이름을 쉼표로 이어 목록 문자열로
    system_prompt = f"""
당신은 온라인 패션 쇼핑몰 고객센터의 반품·교환 접수 담당자입니다.
사용자 메시지는 상담 녹취(전화 또는 채팅) 하나입니다. 끝까지 읽고 접수서에 들어갈 '최종 확정 내용'만 JSON 객체 하나로 출력합니다.

# 출력 형식 (가장 중요)
- 아래 JSON Schema를 만족하는 JSON 객체 하나만 출력합니다. 첫 글자는 {{, 마지막 글자는 }} 입니다.
- 키는 정확히 다음 22개, 모두 포함, 다른 키 금지: {keys_text}
- 설명, 머리말, 코드 블록(```), 태그를 붙이지 않습니다. 키와 문자열은 큰따옴표, 값이 없으면 null(따옴표 없이). "None"/"null"/"" 문자열 금지.
- 금액·수량은 정수(쉼표·단위 없음). 날짜는 "YYYY-MM-DD". enum 항목은 목록의 값과 글자까지 똑같이 씁니다.
JSON Schema: {schema_text}

# 판단 원칙
1. 최종값: 고객이 말을 바꾸거나 상담원이 정정하면 마지막으로 확정된 값을 씁니다. 품절 등으로 불가능해진 요청, 거절된 주장은 반영하지 않습니다.
2. 녹취에 없거나 해당 없는 값은 null. 추측하지 않습니다. 단 아래 규정으로 계산되는 값은 계산합니다.
3. 상담과 무관한 다른 주문·다른 상품·예전 정보는 무시합니다.

# 접수 규정
- 요청유형: "반품"(환불) 또는 "교환". 최종 요청 기준.
- 사유유형: "단순변심", "사이즈불일치", "상품불량", "오배송" 중 하나. 고객 주장과 판매자 확인 결과가 다르면 판매자 확인 결과를 따릅니다.
  (사이즈가 안 맞아 교환/반품 = 사이즈불일치, 마음이 바뀜·생각과 다름 = 단순변심, 다른 색/다른 상품이 옴 = 오배송, 파손·하자 = 상품불량)
- 반품배송비_원: 단순변심·사이즈불일치 = 6000, 상품불량·오배송 = 0.
- 결제금액_원: 반품·교환 '대상 수량'만큼의 실제 결제금액(쿠폰·할인 반영 후). 여러 개 중 일부만이면 그 수량분만.
- 수량: 반품·교환 대상 수량.
- 환불예정금액_원: 반품이면 결제금액_원 - 반품배송비_원, 교환이면 null.
- 교환사이즈·교환색상: 교환일 때만 채우고, 바꾸지 않는 항목은 주문 값을 그대로 씁니다. 반품이면 둘 다 null.
- 상품분류: 다음 영문 카테고리 중 상품에 맞는 하나. {', '.join(CLASS_NAMES)}
  (러닝화·운동화 = Sports Shoes, 스니커즈·캐주얼화 = Casual Shoes, 구두 = Formal Shoes, 셔츠 = Shirts, 티셔츠 = Tshirts, 가방(백) = Handbags, 백팩 = Backpacks)

# 항목 형식
- 접수일: 녹취 머리의 상담·접수 날짜.
- 주문번호: 최종 확인된 주문번호를 영문 대문자와 하이픈 형식으로("에이 이공삼공..." → "A2030...-..."). 다른 주문 번호는 무시.
- 주문자: 이름만(호칭 제외). 연락처: 고객이 남겨 달라고 한 최종 번호를 "010-0000-0000" 형식으로.
- 상품명: 녹취에 나온 상품 이름. 주문사이즈: "270", "M"처럼(사이즈 개념이 없으면 null). 주문색상: 주문한 색상(실제로 잘못 온 색이 아님).
- 회수 주소: 최종 회수지. 회수시도는 "서울특별시", "경기도", "부산광역시"처럼 정식 명칭. 회수시군구는 시와 구가 함께 있으면 "성남시 분당구"처럼 공백으로.
  회수도로명은 도로명만, 회수건물번호는 숫자 문자열("이백삼십오" → "235"), 회수상세주소는 동·층·호를 숫자로("백이동 천삼백십일호" → "102동 1311호", "비동 칠층" → "B동 7층"). 아파트·건물 이름은 상세주소에 넣지 않습니다.
- 회수희망일: 최종 희망 날짜. "모레", "이번 주 금요일" 같은 표현은 접수일 기준으로 실제 날짜를 계산합니다(한 주는 월요일 시작).

출력 전에 속으로 각 값과 계산(배송비, 환불액, 날짜)을 다시 확인하고, JSON 객체만 출력하세요.
""".strip()
    return system_prompt  # 완성한 프롬프트 반환

### [정답안 해설]
- 샘플(전세계약)과 같은 구조: **형식 규칙 → 최종 합의/정정 원칙 → 항목별 정의 → 계산 규칙**.
- 이 문항의 함정: 고객이 말을 바꾸는 사이즈·색상, 상담원이 정정하는 주문번호, **판매자 귀책(불량·오배송)이면 배송비 0원**, 교환이면 환불예정금액 `null`, 회수 주소가 주문 배송지와 다른 경우, 회수희망일 상대 날짜 계산, enum 값(요청유형·사유유형·상품분류) 정확히 쓰기.

## 응답 검증

### 응답에 붙을 수 있는 형식

아래는 **두 항목만 표시한 축약 예시**입니다. 실제 응답에는 Schema의 모든 키가 모두 필요합니다.

**요청할 형식: JSON 객체만 출력**

```text
{"계약체결일": "2030-01-02", "중도금지급일": null}
```

**Markdown 코드 블록(fence)이 붙은 응답**

````text
```json
{"계약체결일": "2030-01-02", "중도금지급일": null}
```
````

백틱 세 개는 코드 블록의 경계이고, 뒤의 `json`은 언어 표시입니다. 둘 다 JSON 객체의 일부가 아닙니다.

**태그가 붙은 응답**

```text
<json>
{"계약체결일": "2030-01-02", "중도금지급일": null}
</json>
```

`<json>` 같은 태그나 “결과입니다”라는 설명이 붙을 수도 있습니다. 프롬프트에서는 이런 장식 없이 객체만 요청하세요.

### 이 문제의 파싱 방식

제공하는 `parse_response()`는 다음 순서로 처리합니다.

1. `find('{')`와 `rfind('}')`로 **첫 `{`부터 마지막 `}`까지** 잘라냅니다.
2. `json.loads()`로 Python 딕셔너리를 만듭니다. JSON의 `null`은 `None`으로 변환됩니다.
3. 모든 키가 모두 있는지, 자료형·숫자 범위·날짜 형식이 맞는지 검사합니다.

코드 블록이나 태그를 따로 인식하는 것은 아닙니다. JSON 바깥에 다른 중괄호가 없으면, 위의 fence·태그나 `json` 머리말은 잘라낸 범위에서 제외됩니다.\
여러 객체를 연달아 출력하거나 바깥 설명에 `{}`가 있으면 파싱이 실패할 수 있습니다. 작은따옴표, Python의 `None`, 마지막 항목 뒤 쉼표 등 잘못된 JSON 문법도 자동으로 고치지 않습니다.

형식 오류가 나면 프롬프트를 수정하세요. 파싱 성공 여부와 추출 내용의 정확성은 별도로 평가합니다.

In [8]:
# 수정 금지

def parse_response(text):  # 응답 문자열을 검사해 사전으로 바꾸는 함수
    """응답의 첫 {부터 마지막 }까지 읽고, 이 문제의 Schema를 검사합니다."""  # 함수 설명(docstring)
    if not isinstance(text, str):  # 문자열이 아니면
        raise ValueError('모델 응답이 문자열이 아닙니다.')  # 오류
    start = text.find('{')  # 첫 '{' 위치
    end = text.rfind('}')  # 마지막 '}' 위치
    if start == -1 or end < start:  # 못 찾았거나 순서가 이상하면
        raise ValueError('응답에서 JSON 객체를 찾지 못했습니다.')  # 오류

    answer = json.loads(text[start:end + 1])  # 그 범위만 잘라 JSON → 파이썬 사전 (null → None)
    properties = response_schema['properties']  # 스키마의 키 규칙
    if not isinstance(answer, dict) or set(answer) != set(properties):  # 사전이 아니거나 키 집합이 스키마와 다르면
        raise ValueError('JSON의 키가 Schema와 다릅니다. 누락되거나 추가된 키를 확인하세요.')  # 오류 (누락·추가 키)

    type_names = {str: 'string', int: 'integer', float: 'number'}  # 파이썬 자료형 → 스키마 타입 이름
    for name, rule in properties.items():  # 키마다 검사
        value = answer[name]  # 값
        if value is None:  # None이면 (아래 검사 생략)
            continue  # 건너뜀
        value_type = type_names.get(type(value))  # 값의 타입 이름
        if value_type == 'integer' and 'number' in rule['type']:  # 정수인데 규칙이 number를 허용하면
            value_type = 'number'  # number로 취급
        if value_type not in rule['type']:  # 허용 타입이 아니면
            raise ValueError(f'{name}: 자료형이 다릅니다. 필요한 형식: {rule["type"]}')  # 자료형 오류
        if isinstance(value, float) and not math.isfinite(value):  # 실수인데 무한대/NaN이면
            raise ValueError(f'{name}: 유한한 숫자여야 합니다.')  # 오류
        if 'minimum' in rule and value < rule['minimum']:  # 최솟값보다 작으면
            raise ValueError(f'{name}: 음수는 허용하지 않습니다.')  # 오류
        if 'enum' in rule and value not in rule['enum']:  # enum 규칙이 있는데 목록에 없는 값이면
            raise ValueError(f'{name}: 허용된 값이 아닙니다. 가능한 값: {rule["enum"]}')  # 오류
        if rule.get('format') == 'date':  # 날짜 형식 항목이면
            message = f'{name}: YYYY-MM-DD 형식의 실제 날짜를 입력하세요. 예: 2030-01-02'  # 오류 문구
            try:  # 시도
                parsed_date = date.fromisoformat(value)  # ISO 날짜로 해석 (2030-02-30 같은 없는 날짜는 실패)
            except ValueError as error:  # 해석 실패하면
                raise ValueError(message) from error  # 날짜 오류
            if parsed_date.isoformat() != value:  # '2030-1-2'처럼 0을 생략한 경우 (되돌린 문자열과 다름)
                raise ValueError(message)  # 날짜 오류
    return answer  # 검사를 통과한 사전 반환

In [9]:
# 수정 금지
# 형식 확인용 예시이며, 실제 문항의 정답이 아닙니다. API는 호출하지 않습니다.

example_answer = {name: None for name in response_schema['properties']}  # 모든 키를 None으로 시작
example_answer['주문번호'] = 'A20300101-0001'  # 예시 주문번호
example_answer['결제금액_원'] = 39000  # 예시 금액
example_answer['요청유형'] = '교환'  # 예시 enum 값
example_answer['접수일'] = '2030-01-02'  # 예시 날짜
example_response = json.dumps(example_answer, ensure_ascii=False)  # JSON 문자열로 바꿈 (None → null)
parsed = parse_response(example_response)  # 검사 함수 통과 확인
print('요청유형:', parsed['요청유형'], '/ 환불예정금액_원:', parsed['환불예정금액_원'])  # 값 확인 출력

요청유형: 교환 / 환불예정금액_원: None


## 5. 예제 추론 및 시스템 프롬프트 저장

먼저 `F3_qa.csv`의 예제 2개로 프롬프트를 확인합니다. 각 예제의 응답 원문과 형식 검사 결과가 셀 아래에 표시됩니다.\
프롬프트를 고쳤다면 함수 셀과 예제 추론 셀을 다시 실행하세요.

`gms_config.pkl`에는 호출에 사용할 OpenAI 객체가 준비되어 있습니다. 아래 코드는 `pickle.load()`로 표준 OpenAI 객체를 복원한 뒤 `client.chat.completions.create()`로 호출합니다. 연결 설정과 모델 정보는 객체에 저장되어 있으므로 별도로 입력하지 않습니다.

**연결 및 모델 정보를 임의로 바꾸거나 별도 주소로 우회를 시도하는 경우, 부정행위로 처리될 수 있습니다. 반드시 제공되는 모델만 사용해야 하며, 3번 문제 풀이 외의 목적으로 사용하는 경우에도 부정행위 처리될 수 있습니다.**

- 시스템 프롬프트는 `outputs/F3_<수험번호>.txt`에 UTF-8로 저장됩니다. 이 파일이 제출할 프롬프트 답안입니다.
- 예제 응답은 셀 출력에서 확인합니다. 형식 검사 통과가 내용의 정답을 뜻하지는 않습니다.
- 호출이나 형식 검사에 실패해도 다음 예제로 진행합니다. 다시 실행하면 현재 프롬프트로 예제를 재호출하고 TXT를 덮어씁니다.

In [10]:
# 수정 금지

with Path('gms_config.pkl').open('rb') as source:  # 연결 객체 파일 열기 (바이너리)
    client = pickle.load(source)  # pickle로 객체 복원
    client.api_key = USER_API_KEY  # API 키 설정

def call_llm(problem, system_prompt):  # 언어모델 한 번 호출 함수
    if not isinstance(system_prompt, str) or not system_prompt.strip():  # 프롬프트가 비었으면
        raise ValueError('system_prompts()에서 비어 있지 않은 시스템 프롬프트 문자열을 반환하세요.')  # 오류
    question = problem['question']  # 녹취 원문
    if not isinstance(question, str) or not question.strip():  # 비었으면
        raise ValueError('문항의 question은 비어 있지 않은 문자열이어야 합니다.')  # 오류
    completion = client.chat.completions.create(  # Chat Completions API 호출
        model=client.model,  # 연결 객체에 저장된 모델 이름
        messages=[  # 메시지 목록
            {'role': 'system', 'content': system_prompt},  # system 역할: 작성한 프롬프트
            {'role': 'user', 'content': question},  # user 역할: 녹취 원문 그대로
        ],
    )
    if not completion.choices or completion.choices[0].finish_reason != 'stop':  # 응답이 없거나 정상 종료(stop)가 아니면 (길이 초과 등)
        raise ValueError('모델 응답이 정상적으로 완료되지 않았습니다.')  # 오류
    text = completion.choices[0].message.content  # 응답 텍스트
    if not isinstance(text, str) or not text.strip():  # 비었으면
        raise ValueError('모델 응답이 비어 있습니다.')  # 오류
    return text  # 응답 텍스트 반환


In [11]:
# 수정 금지

started = time.perf_counter()  # 시작 시각
system_prompt = system_prompts()  # 프롬프트 생성
if not isinstance(system_prompt, str) or not system_prompt.strip():  # 프롬프트가 비었으면
    raise ValueError('system_prompts()에서 비어 있지 않은 시스템 프롬프트 문자열을 반환하세요.')  # 오류

valid_count = 0  # 형식 통과 개수
for index, problem in enumerate(examples):  # 예제마다
    print('\n' + '=' * 60)  # 구분선
    print(f'예제 {index + 1}/{len(examples)} | {problem["task"]}')  # 예제 번호와 작업 이름
    try:  # 시도
        response_text = call_llm(problem, system_prompt)  # 모델 호출
    except Exception as error:  # 호출 중 오류가 나면
        print(f'호출 실패: {type(error).__name__}')  # 오류 종류만 출력하고 다음 예제로
        continue  # 건너뜀

    print('모델 응답:')  # 응답 머리글
    print(response_text)  # 응답 원문 출력
    try:  # 시도
        parse_response(response_text)  # 형식 검사
    except Exception as error:  # 호출 중 오류가 나면
        print(f'형식 검사 실패: {error}')  # 실패 이유 출력
    else:  # 그렇지 않으면 (아래 실행)
        valid_count += 1  # 통과 개수 증가
        print('형식 검사 통과 (내용의 정답 여부를 의미하지 않습니다.)')  # 통과 메시지

output_dir = Path('outputs')  # 결과 저장 폴더
output_dir.mkdir(exist_ok=True)  # 폴더가 없으면 만듦 (있으면 그대로)
output_path = output_dir / f'F3_{student_id}.txt'  # 제출 파일 경로
with output_path.open('w', encoding='utf-8', newline='\n') as output:  # UTF-8, 줄바꿈 \n으로 쓰기
    output.write(system_prompt)  # 프롬프트 문자열만 저장

print(f'\n시스템 프롬프트 저장: {output_path}')  # 저장 경로 출력
print(f'예제 {len(examples)}개 / 형식 검사 통과 {valid_count}개')  # 통과 개수 출력
print(f'Wall-clock: {time.perf_counter() - started:.1f}초')  # 경과 시간


예제 1/2 | 반품·교환 접수 정보 추출
모델 응답:
{"접수일": null, "주문번호": null, "주문자": null, "연락처": null, "상품명": null, "상품분류": null, "주문사이즈": null, "주문색상": null, "수량": null, "결제금액_원": null, "요청유형": null, "사유유형": null, "교환사이즈": null, "교환색상": null, "반품배송비_원": null, "환불예정금액_원": null, "회수시도": null, "회수시군구": null, "회수도로명": null, "회수건물번호": null, "회수상세주소": null, "회수희망일": null}
형식 검사 통과 (내용의 정답 여부를 의미하지 않습니다.)

예제 2/2 | 반품·교환 접수 정보 추출
모델 응답:
{"접수일": null, "주문번호": null, "주문자": null, "연락처": null, "상품명": null, "상품분류": null, "주문사이즈": null, "주문색상": null, "수량": null, "결제금액_원": null, "요청유형": null, "사유유형": null, "교환사이즈": null, "교환색상": null, "반품배송비_원": null, "환불예정금액_원": null, "회수시도": null, "회수시군구": null, "회수도로명": null, "회수건물번호": null, "회수상세주소": null, "회수희망일": null}
형식 검사 통과 (내용의 정답 여부를 의미하지 않습니다.)

시스템 프롬프트 저장: outputs\F3_000000.txt
예제 2개 / 형식 검사 통과 2개
Wall-clock: 0.0초


> **[정답안 전용 셀]** 예제 + 숨은 테스트 1건을 손 정답(`_grading/f3_truth.json`)과 항목별 비교합니다. 실제 키가 있어야 의미가 있습니다.

In [12]:
# [정답안 전용] 손 정답과 비교 (예제 2 + 숨은 테스트 1)
truth = json.load(open('_grading/f3_truth.json', encoding='utf-8'))  # 손 정답 읽기
hidden = load_questions('_grading/f3_hidden.csv')  # 숨은 테스트 읽기
total = correct = 0  # 전체·맞힌 항목 수
for index, problem in enumerate(examples + hidden):  # 예제 + 숨은 테스트 모두
    try:  # 시도
        answer = parse_response(call_llm(problem, system_prompts()))  # 호출 → 형식 검사
    except Exception as error:  # 호출 중 오류가 나면
        print(f'문항 {index + 1}: 호출/형식 실패 - {error}')  # 실패 출력
        continue  # 건너뜀
    expected = truth[index]  # 이 예제의 정답
    wrong = {k: (answer[k], v) for k, v in expected.items() if answer[k] != v}  # 다른 항목만 모음
    total += len(expected); correct += len(expected) - len(wrong)  # 개수 누적
    print(f'문항 {index + 1}: {len(expected) - len(wrong)}/{len(expected)} 일치')  # 문항별 결과
    for key, (got, want) in wrong.items():  # 틀린 항목마다
        print(f'   {key}: 응답 {got!r} / 정답 {want!r}')  # 응답과 정답을 나란히 출력
if total:  # 하나라도 평가했으면
    print(f'필드 정확도: {correct / total:.3f}')  # 필드 정확도 출력

문항 1: 1/22 일치
   접수일: 응답 None / 정답 '2030-05-12'
   주문번호: 응답 None / 정답 'A20300509-0042'
   주문자: 응답 None / 정답 '김도현'
   연락처: 응답 None / 정답 '010-4789-1234'
   상품명: 응답 None / 정답 '에어웨이브 러너2'
   상품분류: 응답 None / 정답 'Sports Shoes'
   주문사이즈: 응답 None / 정답 '270'
   주문색상: 응답 None / 정답 '검정'
   수량: 응답 None / 정답 1
   결제금액_원: 응답 None / 정답 89000
   요청유형: 응답 None / 정답 '교환'
   사유유형: 응답 None / 정답 '사이즈불일치'
   교환사이즈: 응답 None / 정답 '275'
   교환색상: 응답 None / 정답 '검정'
   반품배송비_원: 응답 None / 정답 6000
   회수시도: 응답 None / 정답 '경기도'
   회수시군구: 응답 None / 정답 '성남시 분당구'
   회수도로명: 응답 None / 정답 '판교역로'
   회수건물번호: 응답 None / 정답 '235'
   회수상세주소: 응답 None / 정답 'B동 7층'
   회수희망일: 응답 None / 정답 '2030-05-14'
문항 2: 2/22 일치
   접수일: 응답 None / 정답 '2030-06-03'
   주문번호: 응답 None / 정답 'B20300528-1187'
   주문자: 응답 None / 정답 '이수진'
   연락처: 응답 None / 정답 '010-2233-5566'
   상품명: 응답 None / 정답 '린넨 오버핏 셔츠'
   상품분류: 응답 None / 정답 'Shirts'
   주문사이즈: 응답 None / 정답 'M'
   주문색상: 응답 None / 정답 '화이트'
   수량: 응답 None / 정답 1
   결제금액_원: 응답 None / 정답 42000
   요청유형: 응답 None

## 제출

아래 파일들을 제출하세요.
- 완성한 노트북 `.ipynb`
- 생성된 `outputs/F3_<수험번호>.txt` 파일

파일명에는 첫 셀의 수험번호가 자동으로 들어갑니다. 따로 이름을 바꿀 필요가 없습니다.\
TXT에는 최종 시스템 프롬프트 문자열만 들어 있어야 합니다. 파일명에 맨 위 셀에서 입력한 수험번호가 사용되었는지 확인하세요.

프롬프트를 수정했다면 함수 셀과 예제 추론 셀을 다시 실행해 파일을 갱신하세요.\
예제 응답과 형식 검사 결과는 셀 출력에서 확인할 수 있습니다.